# Backtracking — From Zero to LeetCode

Read top to bottom. By the end you will understand:
- What backtracking actually is
- The universal template (works for 90% of problems)
- How to think: choices, constraints, goal
- 6 classic patterns with full working code

---
## 1. What is Backtracking?

Backtracking = **Try every possibility + Undo bad choices**

Think of it like a maze:
```
You're at START. You try a path.
If it leads to a dead end -> BACKTRACK (go back) -> try another path.
Keep doing this until you find the EXIT.
```

In code:
```
1. Make a choice  (add element, place queen, pick number)
2. Recurse        (go deeper with that choice)
3. Undo the choice (remove element, unplace queen) <- THIS is backtracking
4. Try next choice
```

### Backtracking vs Brute Force
- Brute force: try ALL combinations blindly
- Backtracking: try combinations BUT **prune** (skip) branches that can't work early
- Result: much faster than brute force

### Backtracking vs DP
- DP: overlapping subproblems, store results
- Backtracking: **enumerate all valid solutions**, undo and try next
- Use backtracking when you need ALL solutions or the problem has constraints to check

---
## 2. The Universal Template

Every backtracking problem fits this shape:

```python
def backtrack(current_state, choices_left):
    # BASE CASE: reached goal
    if goal_reached(current_state):
        result.append(copy of current_state)
        return

    for choice in choices_left:
        # CONSTRAINT: skip invalid choices early (pruning)
        if not is_valid(choice):
            continue

        # CHOOSE: make the choice
        make_choice(current_state, choice)

        # EXPLORE: recurse deeper
        backtrack(current_state, updated_choices)

        # UN-CHOOSE: undo the choice (backtrack!)
        undo_choice(current_state, choice)
```

**3 questions to ask for every problem:**
1. **What is the choice?** (what am I picking at each step)
2. **What is the constraint?** (what makes a choice invalid)
3. **What is the goal?** (when do I save the result)

---
## 3. First Example — Subsets (LeetCode 78)

**Problem:** Given `[1,2,3]`, return all possible subsets.

```
Output: [[], [1], [2], [3], [1,2], [1,3], [2,3], [1,2,3]]
```

**Think:**
- Choice: include `nums[i]` or skip it
- Constraint: none (all subsets are valid)
- Goal: every path in the tree is a valid subset

**Decision Tree:**
```
                  []
          [1]           []
       [1,2] [1]     [2]   []
  [1,2,3][1,2][1,3][1] [2,3][2][3][]
```

In [ ]:
def subsets(nums):
    result = []

    def backtrack(start, current):
        result.append(current[:])   # every path = valid subset, save a COPY

        for i in range(start, len(nums)):
            current.append(nums[i])           # CHOOSE
            backtrack(i + 1, current)         # EXPLORE (i+1 to avoid reuse)
            current.pop()                     # UN-CHOOSE (backtrack)

    backtrack(0, [])
    return result


print(subsets([1, 2, 3]))
print(f"Total subsets: {len(subsets([1,2,3]))}")

**Why `current[:]`?**
Lists are passed by reference. If you append `current` directly, all saved results will be the same (empty) list at the end. `current[:]` makes a snapshot copy.

---
## 4. Permutations (LeetCode 46)

**Problem:** Given `[1,2,3]`, return all permutations.

```
Output: [[1,2,3],[1,3,2],[2,1,3],[2,3,1],[3,1,2],[3,2,1]]
```

**Think:**
- Choice: any unused number
- Constraint: can't reuse same number
- Goal: current list has all n numbers

**Difference from subsets:** order matters, use all elements, track `used` set

In [ ]:
def permute(nums):
    result = []

    def backtrack(current, used):
        # GOAL: used all numbers
        if len(current) == len(nums):
            result.append(current[:])
            return

        for num in nums:
            if num in used:           # CONSTRAINT: skip used numbers
                continue
            current.append(num)       # CHOOSE
            used.add(num)
            backtrack(current, used)  # EXPLORE
            current.pop()             # UN-CHOOSE
            used.remove(num)

    backtrack([], set())
    return result


print(permute([1, 2, 3]))
print(f"Total permutations: {len(permute([1,2,3]))}")

---
## 5. Combination Sum (LeetCode 39)

**Problem:** Given candidates `[2,3,6,7]` and target `7`, find all combinations that sum to target. Same number can be used multiple times.

```
Output: [[2,2,3], [7]]
```

**Think:**
- Choice: pick `candidates[i]`
- Constraint: remaining sum can't go below 0
- Goal: remaining == 0
- Key: pass same `i` (not `i+1`) to allow reuse of same number

In [ ]:
def combinationSum(candidates, target):
    result = []

    def backtrack(start, current, remaining):
        if remaining == 0:            # GOAL: exact sum reached
            result.append(current[:])
            return
        if remaining < 0:             # CONSTRAINT: overshot, prune branch
            return

        for i in range(start, len(candidates)):
            current.append(candidates[i])                      # CHOOSE
            backtrack(i, current, remaining - candidates[i])  # EXPLORE (i, not i+1 = reuse allowed)
            current.pop()                                      # UN-CHOOSE

    backtrack(0, [], target)
    return result


print(combinationSum([2, 3, 6, 7], 7))   # [[2,2,3],[7]]
print(combinationSum([2, 3, 5], 8))      # [[2,2,2,2],[2,3,3],[3,5]]

---
## 6. Visualizing the Recursion Tree

Always draw the tree when stuck. Here's `combinationSum([2,3,6,7], 7)` visualized:

In [ ]:
def combinationSum_verbose(candidates, target):
    result = []

    def backtrack(start, current, remaining, depth):
        indent = "  " * depth
        print(f"{indent}backtrack(start={start}, current={current}, remaining={remaining})")

        if remaining == 0:
            print(f"{indent}  -> FOUND: {current}")
            result.append(current[:])
            return
        if remaining < 0:
            print(f"{indent}  -> PRUNED (overshot)")
            return

        for i in range(start, len(candidates)):
            current.append(candidates[i])
            backtrack(i, current, remaining - candidates[i], depth + 1)
            current.pop()

    backtrack(0, [], target, 0)
    return result

combinationSum_verbose([2, 3], 5)

---
## 7. Palindrome Partitioning (LeetCode 131)

**Problem:** Partition string `"aab"` such that every substring is a palindrome.

```
Output: [["a","a","b"], ["aa","b"]]
```

**Think:**
- Choice: cut at position `i`, take `s[start:i+1]`
- Constraint: the cut substring must be a palindrome
- Goal: `start == len(s)` (processed whole string)

In [ ]:
def partition(s):
    result = []

    def is_palindrome(sub):
        return sub == sub[::-1]

    def backtrack(start, current):
        if start == len(s):           # GOAL: used whole string
            result.append(current[:])
            return

        for end in range(start + 1, len(s) + 1):
            substring = s[start:end]
            if not is_palindrome(substring):  # CONSTRAINT: skip non-palindromes
                continue
            current.append(substring)         # CHOOSE
            backtrack(end, current)           # EXPLORE
            current.pop()                     # UN-CHOOSE

    backtrack(0, [])
    return result


print(partition("aab"))    # [["a","a","b"],["aa","b"]]
print(partition("racecar"))

---
## 8. N-Queens (LeetCode 51) — Classic Backtracking

**Problem:** Place N queens on NxN board so no two queens attack each other.

Queens attack on same row, column, or diagonal.

```
N=4 solution:
. Q . .
. . . Q
Q . . .
. . Q .
```

**Think:**
- Place one queen per row (row by row)
- Choice: which column to place queen in row `r`
- Constraint: column not used, diagonals not attacked
- Goal: placed queens in all N rows

In [ ]:
def solveNQueens(n):
    result = []
    cols = set()       # columns with queens
    diag1 = set()      # row - col  (top-left to bottom-right diagonal)
    diag2 = set()      # row + col  (top-right to bottom-left diagonal)

    board = [["." for _ in range(n)] for _ in range(n)]

    def backtrack(row):
        if row == n:                  # GOAL: placed queen in every row
            result.append(["".join(r) for r in board])
            return

        for col in range(n):
            # CONSTRAINT: column or diagonal already attacked
            if col in cols or (row-col) in diag1 or (row+col) in diag2:
                continue

            # CHOOSE
            board[row][col] = "Q"
            cols.add(col)
            diag1.add(row - col)
            diag2.add(row + col)

            backtrack(row + 1)        # EXPLORE

            # UN-CHOOSE
            board[row][col] = "."
            cols.remove(col)
            diag1.remove(row - col)
            diag2.remove(row + col)

    backtrack(0)
    return result


solutions = solveNQueens(4)
print(f"N=4: {len(solutions)} solutions")
for sol in solutions:
    for row in sol: print(row)
    print()

---
## 9. Word Search (LeetCode 79) — Grid Backtracking

**Problem:** Given a grid and a word, check if the word exists as a path in the grid (up/down/left/right).

**Think:**
- Choice: move to adjacent cell
- Constraint: in bounds, not already visited, matches word character
- Goal: matched all characters of the word

In [ ]:
def exist(board, word):
    rows, cols = len(board), len(board[0])

    def backtrack(r, c, index):
        if index == len(word):            # GOAL: matched whole word
            return True
        if r < 0 or r >= rows or c < 0 or c >= cols:  # out of bounds
            return False
        if board[r][c] != word[index]:    # CONSTRAINT: wrong character
            return False
        if board[r][c] == '#':            # already visited
            return False

        temp = board[r][c]
        board[r][c] = '#'                 # CHOOSE: mark visited

        # EXPLORE all 4 directions
        found = (backtrack(r+1, c, index+1) or
                 backtrack(r-1, c, index+1) or
                 backtrack(r, c+1, index+1) or
                 backtrack(r, c-1, index+1))

        board[r][c] = temp                # UN-CHOOSE: restore
        return found

    for r in range(rows):
        for c in range(cols):
            if backtrack(r, c, 0):        # try starting from every cell
                return True
    return False


board = [
    ['A','B','C','E'],
    ['S','F','C','S'],
    ['A','D','E','E']
]
print(exist(board, "ABCCED"))   # True
print(exist(board, "SEE"))      # True
print(exist(board, "ABCB"))     # False

---
## 10. LeetCode Backtracking Patterns Cheatsheet

| Problem Type | start param? | reuse allowed? | used set? | Goal condition |
|---|---|---|---|---|
| Subsets | Yes (`i+1`) | No | No | Every node |
| Combinations | Yes (`i+1`) | No | No | `len == k` |
| Permutations | No | No | Yes | `len == n` |
| Combination Sum | Yes (`i`) | Yes (same element) | No | `remaining == 0` |
| Palindrome partition | Yes (end index) | No | No | `start == len(s)` |
| N-Queens | Row number | No | cols/diag sets | `row == n` |
| Word Search | r, c, index | No | mark visited | `index == len(word)` |

---
## Summary

```
WHAT IS BACKTRACKING?
  Try a choice -> recurse -> undo choice -> try next
  = DFS on a decision tree + pruning invalid branches early

TEMPLATE (memorize this):
  def backtrack(state):
      if goal: save result; return
      for choice in choices:
          if invalid: continue          <- PRUNE
          make choice                   <- CHOOSE
          backtrack(new state)          <- EXPLORE
          undo choice                   <- UN-CHOOSE

3 QUESTIONS:
  1. What is the CHOICE?      (what am I picking)
  2. What is the CONSTRAINT?  (what makes it invalid -> prune early)
  3. What is the GOAL?        (when to save result)

KEY DETAILS:
  Always append current[:] not current (copy!)
  Use start index to avoid duplicates in subsets/combinations
  Use used set to avoid reuse in permutations
  Pass same i (not i+1) if reuse of same element is allowed
```

Now open `1-leetcode-backtracking.ipynb` — every problem will make sense!